In [1]:
import os

In [2]:
%pwd

'e:\\Personal\\SmartEEDigital\\Study\\MLOps\\E2E-Machine-Learning-Project-with-MLFlow\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'e:\\Personal\\SmartEEDigital\\Study\\MLOps\\E2E-Machine-Learning-Project-with-MLFlow'

In [5]:
os.environ["MLFLOW_TRACKING_URI"]="https://dagshub.com/haluk.a.turan/E2E-Machine-Learning-Project-with-MLFlow.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"]="haluk.a.turan"
os.environ["MLFLOW_TRACKING_PASSWORD"]="767df0969822c86fd14ee63e1995613b4c4113c6"

In [14]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelEvaluationConfig:
    root_dir: Path
    test_data_path: Path
    model_path: Path
    all_params: dict
    metric_file_name: Path
    target_column: str
    mlflow_uri: str

In [15]:
from MLProject.constants import *
from MLProject.utils.common import read_yaml, create_directories, save_json

In [ ]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH,
        schema_filepath=SCHEMA_FILE_PATH
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        config = self.config.model_evaluation
        params = self.params.ElasticNet
        schema = self.schema.TARGET_COLUMN

        create_directories([config.root_dir])

        model_evaluation_config = ModelEvaluationConfig(
            root_dir = config.root_dir,
            test_data_path = config.test_data_path,
            model_path = config.model_path,
            all_params = params,
            metric_file_name = config.metric_file_name,
            target_column = schema.name,
            mlflow_uri = "https://dagshub.com/haluk.a.turan/E2E-Machine-Learning-Project-with-MLFlow.mlflow"
        )

        return model_evaluation_config

In [17]:
import os
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from urllib.parse import urlparse
import mlflow
import mlflow.sklearn
import numpy as np
import joblib

In [18]:
class ModelEvaluation:
    def __init__(self, config: ModelEvaluationConfig):
        self.config = config

    def eval_metrics(self, actual, pred):
        rmse = np.sqrt(mean_squared_error(actual, pred))
        mae = mean_absolute_error(actual, pred)
        r2 = r2_score(actual, pred)

        return rmse, mae, r2

    def log_into_mlflow(self):
        test_data = pd.read_csv(self.config.test_data_path)
        model = joblib.load(self.config.model_path)

        test_x = test_data.drop([self.config.target_column], axis=1)
        test_y = test_data[[self.config.target_column]]

        mlflow.set_registry_uri(self.config.mlflow_uri)
        tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme

        with mlflow.start_run():
            predicted_qualities = model.predict(test_x)

            (rmse, mae, r2) = self.eval_metrics(test_y, predicted_qualities)

            # Saving metrics as local
            scores = {"rmse": rmse, "mae": mae, "r2": r2}
            save_json(path=Path(self.config.metric_file_name), data=scores)

            mlflow.log_params(self.config.all_params)

            mlflow.log_metric("rmse", rmse)
            mlflow.log_metric("r2", r2)
            mlflow.log_metric("mae", mae)

            # Model registry does not work with file store
            if tracking_url_type_store != "file":
                # Register the model
                # There are other ways to use the Model Registry, which depends on the use case,
                # please refer to the doc for more imformation:
                # https://mlflow.org/docs/latest/model-registry.html#api-workfow
                mlflow.sklearn.log_model(model, "model", registered_model_name="ElasticnetModel")
            else:
                mlflow.sklearn.log_model(model, "model")


In [19]:
try:
    config = ConfigurationManager()
    model_evaluation_config = config.get_model_evaluation_config()
    model_evaluation_config = ModelEvaluation(config=model_evaluation_config)
    model_evaluation_config.log_into_mlflow()
except Exception as e:
    raise e

[2026-03-16 10:12:47,864] INFO - common - yaml file: config\config.yaml loaded successfully
[2026-03-16 10:12:47,865] INFO - common - yaml file: params.yaml loaded successfully
[2026-03-16 10:12:47,867] INFO - common - yaml file: schema.yaml loaded successfully
[2026-03-16 10:12:47,868] INFO - common - created directory at: artifacts
[2026-03-16 10:12:47,869] INFO - common - created directory at: artifacts/model_evaluation
[2026-03-16 10:12:48,801] INFO - common - json file saved at: artifacts\model_evaluation\metrics.json


2026/03/16 10:12:49 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/16 10:12:51 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'ElasticnetModel' already exists. Creating a new version of this model...
2026/03/16 10:12:59 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: ElasticnetModel, version 2
Created version '2' of model 'ElasticnetModel'.


🏃 View run youthful-dog-170 at: https://dagshub.com/haluk.a.turan/E2E-Machine-Learning-Project-with-MLFlow.mlflow/#/experiments/0/runs/b9281919ee294dceba9a1df587953961
🧪 View experiment at: https://dagshub.com/haluk.a.turan/E2E-Machine-Learning-Project-with-MLFlow.mlflow/#/experiments/0
